In [4]:
import gradio as gr
import fitz  # PyMuPDF (fallback for OCR)
from PyPDF2 import PdfReader
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass
import json
from datetime import datetime
import tempfile
import os
from llama_cpp import Llama
from types import SimpleNamespace

# LlamaIndex imports
from llama_index.core import Document, VectorStoreIndex
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.vector_stores import MetadataFilters, MetadataFilter, FilterOperator

# ============================================
# STEP 1: Core Model & Embedding Configuration
# ============================================

llm = Llama.from_pretrained(
    repo_id="bartowski/Mistral-7B-Instruct-v0.3-GGUF",
    filename="Mistral-7B-Instruct-v0.3-Q5_K_S.gguf",
    n_gpu_layers=-1,
    n_ctx=5096,
    verbose=False
)

class LocalModelWrapper:
    def __init__(self, llm_instance):
        self.llm = llm_instance

    def generate_content(self, prompt: str):
        output = self.llm.create_chat_completion(
            messages=[{"role": "user", "content": prompt}],
            max_tokens=512,
            temperature=0.0
        )
        text = output["choices"][0]["message"]["content"]
        return SimpleNamespace(text=text)

    def __call__(self, prompt: str) -> str:
        return self.generate_content(prompt).text

gemini_model = LocalModelWrapper(llm)

# Initialize embedding models
embed_model = SentenceTransformer("BAAI/bge-small-en-v1.5")
llama_embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

print("Imports and configuration complete.")

# ============================================
# STEP 2: Data Structures for Document Management
# ============================================

@dataclass
class PageInfo:
    """Stores information about a single page"""
    page_num: int
    text: str
    doc_type: Optional[str] = None
    page_in_doc: int = 0

@dataclass
class LogicalDocument:
    """Represents a logical document within a PDF"""
    doc_id: str
    doc_type: str
    page_start: int
    page_end: int
    text: str
    chunks: List['ChunkMetadata'] = None

@dataclass
class ChunkMetadata:
    """Rich metadata for each chunk"""
    chunk_id: str
    doc_id: str
    doc_type: str
    chunk_index: int
    page_start: int
    page_end: int
    text: str
    embedding: Optional[np.ndarray] = None

# ============================================
# STEP 3: Document Intelligence (from gradio_RAG_chat)
# ============================================

VALID_DOC_TYPES = [
    "Cover Letter", "Certificate Of Quality", "Packaging Specification",
    "Bse/Tse Declaration", "Material Description", "Supplier Qualification",
    "Chain Of Custody", "Other"
]

def clean_doc_type(response: str) -> str:
    """Clean up LLM response to extract a valid doc_type label."""
    cleaned = response.strip().replace('"', '').replace('`', '').replace('*', '').lower().replace(".", "").strip()
    cleaned_title = cleaned.title()
    for label in VALID_DOC_TYPES:
        if label.lower() in cleaned.lower():
            return label
    return cleaned_title if cleaned_title in VALID_DOC_TYPES else "Other"

def classify_document_type(text: str) -> str:
    """
    Classify the document type based on its full page content.
    Includes explicit title checks + exact LLM prompt from gradio_RAG_chat
    so documents like Chain of Custody at the end of a blob PDF are never missed.
    """
    if not text or not text.strip():
        return "Other"

    # Fast, high-precision check for explicit document headers/titles on the page
    lower_text = text.lower()
    if "chain of custody" in lower_text:
        return "Chain Of Custody"
    if "bse/tse" in lower_text or "transmissible spongiform" in lower_text or "animal origin" in lower_text:
        return "Bse/Tse Declaration"
    if "packaging specification" in lower_text:
        return "Packaging Specification"
    if "certificate of quality" in lower_text or "certificate of analysis" in lower_text or "certificate of conformance" in lower_text:
        return "Certificate Of Quality"
    if "supplier qualification" in lower_text:
        return "Supplier Qualification"
    if "material description" in lower_text:
        return "Material Description"
    if "to whom it may concern" in lower_text or "dear customer" in lower_text:
        return "Cover Letter"

    # Full-page LLM classification (matching gradio_RAG_chat exactly)
    prompt = f"""
    You are a pharmaceutical document classifier. Based on the page
    content below, classify it into ONE of these document types:

    - Cover Letter: A formal letter (often starting with "To Whom It
      May Concern") discussing product information or storage conditions.
    - Certificate of Quality: Contains lot numbers, manufacture dates,
      expiration dates, and test results (autoclave, gamma irradiation).
    - Packaging Specification: Describes packaging components, materials,
      part numbers, and configuration change history.
    - BSE/TSE Declaration: A declaration about animal-origin materials
      and transmissible spongiform encephalopathy compliance.
    - Material Description: Lists materials of construction, sterilization
      compatibility, and physical properties of a product.
    - Supplier Qualification: Contains supplier audit history,
      certifications (ISO 9001, ISO 13485), and approved product lists.
    - Chain of Custody: Lists manufactured assemblies, traceability
      information, and the manufacturing-to-shipment flow.
    - Other: Use ONLY if the content does not match any of the above.

    Page Content:
    {text}

    Respond with ONLY the document type name. No explanation.
    """
    try:
        response = gemini_model(prompt)
        return clean_doc_type(response)
    except Exception as e:
        print(f"Classification error: {e}")
        return "Other"

def is_same_document(prev_text: str, curr_text: str, doc_type: str = None) -> bool:
    """
    Exact boundary detection logic from gradio_RAG_chat.
    Compares the full text of two consecutive pages classified as the same doc_type.
    """
    prompt = f"""You are comparing the full text of two consecutive pages that were both classified as '{doc_type}'.
Determine whether Current Page is a continuation of the exact same document from Previous Page, or a separate standalone document.

Read the full pages carefully and apply these content rules:
- Answer 'No' (separate documents) if both pages are Certificates of Quality for different Lot Numbers or different products.
- Answer 'No' (separate documents) if Current Page restarts at Section 1 or introduces a brand new standalone record.
- Answer 'Yes' (same document) ONLY if Current Page continues the unfinished sections, tables, or text of the exact same document from Previous Page (for example, Previous Page ends at Section 3 and Current Page starts at Section 4).

Previous Page Full Text:
{prev_text}

Current Page Full Text:
{curr_text}

Do these two pages belong to the SAME individual document? Answer ONLY 'Yes' or 'No'."""

    try:
        response = gemini_model(prompt)
        return response.strip().lower().startswith("yes")
    except Exception as e:
        print(f"Boundary detection error: {e}")
        return False

def predict_doc_type_for_query(query: str) -> str:
    """Exact query routing prompt from gradio_RAG_chat."""
    prompt = f"""
    You are an intelligent assistant that routes user queries to the most relevant pharmaceutical document.

    User query: "{query}"

    Which document type is most likely to contain the answer?
    Choose ONLY ONE from: Cover Letter, Certificate of Quality, Packaging Specification, BSE/TSE Declaration,
    Material Description, Supplier Qualification, Chain of Custody, Other.

    Respond with ONLY the document type name. No explanation.
    """
    try:
        response = gemini_model(prompt)
        return clean_doc_type(response)
    except Exception as e:
        print(f"Query routing error: {e}")
        return "Other"

# ============================================
# STEP 4: PDF Processing Pipeline (gradio_RAG_chat + Enhanced Metadata)
# ============================================

def extract_and_analyze_pdf(pdf_file) -> Tuple[List[PageInfo], List[LogicalDocument]]:
    """
    Uses PdfReader first (identical to gradio_RAG_chat) so page text layout
    matches the expected format, with PyMuPDF + OCR fallback if a page is scanned.
    """
    print("Starting PDF extraction and analysis...")
    file_path = pdf_file.name if hasattr(pdf_file, "name") else pdf_file

    reader = PdfReader(file_path)
    raw_pages = [page.extract_text() or "" for page in reader.pages]

    # OCR fallback if any page has no extractable text
    pages_info: List[PageInfo] = []
    for i, text in enumerate(raw_pages):
        if not text.strip():
            try:
                doc = fitz.open(file_path)
                page = doc[i]
                text = page.get_text()
                if not text.strip():
                    pix = page.get_pixmap()
                    img_data = pix.tobytes("png")
                    from PIL import Image
                    import pytesseract
                    import io
                    img = Image.open(io.BytesIO(img_data))
                    text = pytesseract.image_to_string(img)
                doc.close()
            except Exception as e:
                print(f"  Page {i}: Fallback extraction failed - {e}")
                text = ""
        pages_info.append(PageInfo(page_num=i, text=text))

    if not pages_info:
        raise ValueError("No text could be extracted from PDF")

    # Classify every page and detect boundaries using gradio_RAG_chat logic
    metadata_store = []
    page_counter = 0

    for i, page_obj in enumerate(pages_info):
        curr_doc_type = classify_document_type(page_obj.text)

        if i == 0:
            page_counter = 0
        else:
            prev_text = pages_info[i - 1].text
            prev_doc_type = metadata_store[i - 1]["doc_type"]

            if curr_doc_type == prev_doc_type:
                same = is_same_document(prev_text, page_obj.text, curr_doc_type)
            else:
                same = False

            if same:
                page_counter += 1
            else:
                page_counter = 0

        page_obj.doc_type = curr_doc_type
        page_obj.page_in_doc = page_counter

        metadata_store.append({
            "page": i,
            "text": page_obj.text,
            "doc_type": curr_doc_type,
            "page_in_doc": page_counter
        })
        print(f"  Page {i + 1}: {curr_doc_type} (page_in_doc={page_counter})")

    # Group into logical documents (exact loop from gradio_RAG_chat)
    raw_logical_docs = []
    current_doc = {"text": "", "doc_type": None, "page_start": 0}

    for page in metadata_store:
        if page["page_in_doc"] == 0 and current_doc["text"]:
            current_doc["page_end"] = page["page"] - 1
            raw_logical_docs.append(current_doc)
            current_doc = {"text": "", "doc_type": None, "page_start": page["page"]}

        current_doc["text"] += "\n\n" + page["text"]
        current_doc["doc_type"] = page["doc_type"]

    if metadata_store:
        current_doc["page_end"] = metadata_store[-1]["page"]
        raw_logical_docs.append(current_doc)

    logical_docs: List[LogicalDocument] = []
    for idx, doc_dict in enumerate(raw_logical_docs):
        logical_docs.append(
            LogicalDocument(
                doc_id=f"doc_{idx}",
                doc_type=doc_dict["doc_type"],
                page_start=doc_dict["page_start"],
                page_end=doc_dict["page_end"],
                text=doc_dict["text"].strip()
            )
        )

    print(f"Identified {len(logical_docs)} logical documents:")
    for ld in logical_docs:
        print(f"   - {ld.doc_type}: Pages {ld.page_start + 1}-{ld.page_end + 1}")

    return pages_info, logical_docs

# ============================================
# STEP 5: Intelligent Chunking with Metadata Preservation
# ============================================

def chunk_document_with_metadata(logical_doc: LogicalDocument,
                                 chunk_size: int = 500,
                                 overlap: int = 100) -> List[ChunkMetadata]:
    chunks_metadata = []
    words = logical_doc.text.split()

    if len(words) <= chunk_size:
        chunks_metadata.append(
            ChunkMetadata(
                chunk_id=f"{logical_doc.doc_id}_chunk_0",
                doc_id=logical_doc.doc_id,
                doc_type=logical_doc.doc_type,
                chunk_index=0,
                page_start=logical_doc.page_start,
                page_end=logical_doc.page_end,
                text=logical_doc.text
            )
        )
    else:
        stride = max(1, chunk_size - overlap)
        for i, start_idx in enumerate(range(0, len(words), stride)):
            end_idx = min(start_idx + chunk_size, len(words))
            chunk_text = " ".join(words[start_idx:end_idx])

            chunk_position = start_idx / len(words)
            page_range = logical_doc.page_end - logical_doc.page_start
            relative_page = int(chunk_position * page_range)
            chunk_page_start = logical_doc.page_start + relative_page
            chunk_page_end = min(chunk_page_start + 1, logical_doc.page_end)

            chunks_metadata.append(
                ChunkMetadata(
                    chunk_id=f"{logical_doc.doc_id}_chunk_{i}",
                    doc_id=logical_doc.doc_id,
                    doc_type=logical_doc.doc_type,
                    chunk_index=i,
                    page_start=chunk_page_start,
                    page_end=chunk_page_end,
                    text=chunk_text
                )
            )
            if end_idx >= len(words):
                break

    return chunks_metadata

def process_all_documents(logical_docs: List[LogicalDocument]) -> List[ChunkMetadata]:
    all_chunks = []
    for logical_doc in logical_docs:
        chunks = chunk_document_with_metadata(logical_doc)
        logical_doc.chunks = chunks
        all_chunks.extend(chunks)
    return all_chunks

# ============================================
# STEP 6: Hybrid LlamaIndex + FAISS Retriever
# ============================================

class IntelligentRetriever:
    """
    Uses LlamaIndex VectorStoreIndex with MetadataFilters (from gradio_RAG_chat)
    plus FAISS indices for fast top-k similarity scoring.
    """

    def __init__(self):
        self.index = None
        self.llama_index = None
        self.chunks_metadata: List[ChunkMetadata] = []
        self.doc_type_indices = {}

    def build_indices(self, chunks_metadata: List[ChunkMetadata], logical_docs: List[LogicalDocument]):
        self.chunks_metadata = chunks_metadata
        self.doc_type_indices = {}

        # Build FAISS indices
        texts = [chunk.text for chunk in chunks_metadata]
        embeddings = embed_model.encode(texts, show_progress_bar=False, normalize_embeddings=True)
        embeddings = np.array(embeddings, dtype="float32")

        for i, chunk in enumerate(chunks_metadata):
            chunk.embedding = embeddings[i]

        dim = embeddings.shape[1]
        self.index = faiss.IndexFlatIP(dim)
        self.index.add(embeddings)

        doc_types = set(chunk.doc_type for chunk in chunks_metadata)
        for doc_type in doc_types:
            type_indices = [i for i, c in enumerate(chunks_metadata) if c.doc_type == doc_type]
            if type_indices:
                type_index = faiss.IndexFlatIP(dim)
                type_index.add(embeddings[type_indices])
                self.doc_type_indices[doc_type] = {
                    "index": type_index,
                    "mapping": type_indices
                }

        # Build LlamaIndex VectorStoreIndex on logical documents (matching gradio_RAG_chat)
        chunked_documents = [
            Document(
                text=doc.text,
                metadata={
                    "doc_id": doc.doc_id,
                    "doc_type": doc.doc_type,
                    "page_start": doc.page_start,
                    "page_end": doc.page_end
                }
            )
            for doc in logical_docs
        ]
        self.llama_index = VectorStoreIndex.from_documents(chunked_documents, embed_model=llama_embed_model)

    def retrieve(self, query: str, k: int = 4,
                 filter_doc_type: Optional[str] = None,
                 auto_route: bool = True) -> Tuple[List[Tuple[ChunkMetadata, float]], str]:
        """Retrieve relevant chunks using metadata filtering and automatic fallback."""
        target_type = None
        if filter_doc_type and filter_doc_type != "All":
            target_type = filter_doc_type
            routed_label = f"Filter: {target_type}"
        elif auto_route:
            target_type = predict_doc_type_for_query(query)
            print(f"Predicted document type: {target_type}")
            routed_label = f"Auto-Routed: {target_type}"
        else:
            routed_label = "All Documents"

        query_embedding = np.array(
            embed_model.encode([query], normalize_embeddings=True), dtype="float32"
        )

        # Search specific document type index if available
        if target_type and target_type in self.doc_type_indices:
            type_data = self.doc_type_indices[target_type]
            search_k = min(k, len(type_data["mapping"]))
            D, I = type_data["index"].search(query_embedding, search_k)
            chunk_indices = [type_data["mapping"][i] for i in I[0] if i != -1]
            scores = [float(max(0.0, s)) for s in D[0][:len(chunk_indices)]]
        else:
            # Fallback to global index if predicted doc_type isn't present or "All" was selected
            if target_type and target_type != "Other":
                routed_label += " (Fallback to All)"
            search_k = min(k, len(self.chunks_metadata))
            D, I = self.index.search(query_embedding, search_k)
            chunk_indices = [i for i in I[0] if i != -1]
            scores = [float(max(0.0, s)) for s in D[0][:len(chunk_indices)]]

        results = [(self.chunks_metadata[i], scores[idx]) for idx, i in enumerate(chunk_indices)]
        return results, routed_label

# ============================================
# STEP 7: Enhanced Document Store
# ============================================

class EnhancedDocumentStore:
    def __init__(self):
        self.pages_info: List[PageInfo] = []
        self.logical_docs: List[LogicalDocument] = []
        self.chunks_metadata: List[ChunkMetadata] = []
        self.retriever = IntelligentRetriever()
        self.is_ready = False
        self.processing_stats = {}
        self.last_results_dict = {}

    def process_pdf(self, pdf_file, filename: str = "document.pdf"):
        self.is_ready = False
        start_time = datetime.now()

        try:
            self.pages_info, self.logical_docs = extract_and_analyze_pdf(pdf_file)
            self.chunks_metadata = process_all_documents(self.logical_docs)
            self.retriever.build_indices(self.chunks_metadata, self.logical_docs)

            process_time = (datetime.now() - start_time).total_seconds()
            unique_types = []
            for doc in self.logical_docs:
                if doc.doc_type not in unique_types:
                    unique_types.append(doc.doc_type)

            self.processing_stats = {
                "filename": filename,
                "total_pages": len(self.pages_info),
                "documents_found": len(self.logical_docs),
                "total_chunks": len(self.chunks_metadata),
                "document_types": unique_types,
                "processing_time": f"{process_time:.1f}s"
            }
            self.is_ready = True
            return True, self.processing_stats
        except Exception as e:
            return False, {"error": str(e)}

    def query(self, question: str, filter_type: Optional[str] = None,
              auto_route: bool = True, k: int = 4) -> Dict:
        if not self.is_ready:
            return {
                "answer": "⚠️ Please upload a PDF file first.",
                "sources": [],
                "filter_used": "none"
            }

        retrieved, routed_label = self.retriever.retrieve(
            question, k=k, filter_doc_type=filter_type, auto_route=auto_route
        )

        if not retrieved:
            return {
                "answer": "Could not find relevant information in the uploaded document.",
                "sources": [],
                "filter_used": routed_label
            }

        # Format retrieved text and prompt (clean answer style from gradio_RAG_chat + citations)
        retrieved_text = "\n\n---\n\n".join([
            f"[Document: {c.doc_type} | Pages: {c.page_start + 1}-{c.page_end + 1}]\n{c.text}"
            for c, _ in retrieved
        ])

        prompt = f"""Based on the following text, please answer the user's question.

Text:
{retrieved_text}

User Question: {question}

Provide a clear, specific answer based only on the text provided."""

        answer = gemini_model(prompt).strip()

        sources = [
            f"{c.doc_type} (Pages {c.page_start + 1}-{c.page_end + 1})"
            for c, _ in retrieved
        ]
        # Deduplicate sources while preserving order
        unique_sources = list(dict.fromkeys(sources))

        # Build Step 14 structured dictionary for JSON download
        self.last_results_dict = {
            "query": question,
            "predicted_doc_type": routed_label,
            "matched_chunks": [
                {
                    "text": c.text,
                    "score": round(score, 4),
                    "metadata": {
                        "doc_id": c.doc_id,
                        "doc_type": c.doc_type,
                        "page_start": c.page_start,
                        "page_end": c.page_end
                    }
                }
                for c, score in retrieved
            ],
            "answer": answer
        }

        return {
            "answer": answer,
            "sources": unique_sources,
            "filter_used": routed_label
        }

# ============================================
# STEP 8: Minimalistic Gradio UI (Auto-Process on Upload)
# ============================================

doc_store = EnhancedDocumentStore()

def process_pdf_ui(file):
    global doc_store
    if file is None:
        doc_store = EnhancedDocumentStore()
        return (
            "Waiting for PDF...",
            gr.update(choices=["All"], value="All"),
            "No document processed yet."
        )

    file_path = file.name if hasattr(file, "name") else file
    filename = os.path.basename(file_path)
    success, stats = doc_store.process_pdf(file_path, filename=filename)

    if not success:
        return (
            f"❌ Error: {stats.get('error', 'Unknown error')}",
            gr.update(choices=["All"], value="All"),
            "Error processing document."
        )

    status_text = (
        f"✅ Processed {stats['total_pages']} pages into "
        f"{stats['documents_found']} logical documents ({stats['processing_time']})."
    )

    # Build a clean summary list of all detected documents
    breakdown_lines = [
        f"{idx + 1}. **{doc.doc_type}** — Pages {doc.page_start + 1} to {doc.page_end + 1} ({len(doc.chunks)} chunk(s))"
        for idx, doc in enumerate(doc_store.logical_docs)
    ]
    breakdown_md = "\n".join(breakdown_lines)

    dropdown_choices = ["All"] + stats["document_types"]
    return (
        status_text,
        gr.update(choices=dropdown_choices, value="All"),
        breakdown_md
    )

def handle_chat(message, history, doc_filter, auto_route, top_k):
    history = history or []
    if not message or not message.strip():
        return history, ""

    if not doc_store.is_ready:
        history.append({"role": "user", "content": message})
        history.append({"role": "assistant", "content": "⚠️ Please upload a PDF file first."})
        return history, ""

    filter_val = None if doc_filter == "All" else doc_filter
    result = doc_store.query(
        message,
        filter_type=filter_val,
        auto_route=auto_route and (filter_val is None),
        k=int(top_k)
    )

    answer = result["answer"]
    if result.get("sources"):
        answer += f"\n\n*Sources: {', '.join(result['sources'])} ({result['filter_used']})*"

    history.append({"role": "user", "content": message})
    history.append({"role": "assistant", "content": answer})
    return history, ""

def dict_out():
    """Writes the Step 14 dictionary to a JSON file and returns the file path for download."""
    output_path = os.path.join(tempfile.gettempdir(), "rag_results.json")
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(doc_store.last_results_dict, f, indent=2)
    return output_path

def download_chat_history(history):
    """Writes the current Gradio chat history to a JSON file and returns the file path for download."""
    output_path = os.path.join(tempfile.gettempdir(), "chat_history.json")
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(history or [], f, indent=2)
    return output_path

# Build Minimalist Gradio UI matching gradio_RAG_chat (Auto-Processing)
with gr.Blocks(title="Pharmaceutical Document Q&A") as demo:
    gr.Markdown("### 📄 Pharmaceutical Document Q&A")

    with gr.Row():
        # Left Column: Chat (same layout & proportions as gradio_RAG_chat)
        with gr.Column(scale=2):
            chatbot = gr.Chatbot(label="Chat History", height=380)
            user_query = gr.Textbox(
                placeholder="Ask a question about your document...",
                label="Your Question"
            )
            with gr.Row():
                send_btn = gr.Button("📤 Send", variant="primary")
                clear_btn = gr.Button("🗑️ Clear Chat")

        # Right Column: Auto-Upload, Status, and Simple Dropdowns
        with gr.Column(scale=1):
            pdf_input = gr.File(label="📄 Upload PDF (Auto-Processes on Upload)", file_types=[".pdf"])
            status_box = gr.Textbox(label="Status", interactive=False, value="Waiting for PDF...")

            # Minimalist Dropdown Menus (Accordions + Document Filter Dropdown)
            with gr.Accordion("⚙️ Retrieval Options & Detected Documents", open=False):
                doc_filter = gr.Dropdown(
                    choices=["All"],
                    value="All",
                    label="Filter by Document Type"
                )
                auto_route = gr.Checkbox(
                    value=True,
                    label="Auto-Route Query to Best Document Type"
                )
                top_k = gr.Dropdown(
                    choices=[2, 4, 6, 8],
                    value=4,
                    label="Chunks to Retrieve"
                )
                gr.Markdown("**Detected Documents in PDF:**")
                detected_docs_md = gr.Markdown("No document processed yet.")

            with gr.Accordion("📥 Export & Downloads", open=False):
                download_dict_btn = gr.Button("📥 Download Query Details & Metadata", size="sm")
                download_chat_btn = gr.Button("💬 Download Chat History", size="sm")
                download_file_output = gr.File(label="Download File", interactive=False)

    # Event Wiring — Auto-process immediately when a PDF is uploaded or cleared
    pdf_input.change(
        fn=process_pdf_ui,
        inputs=pdf_input,
        outputs=[status_box, doc_filter, detected_docs_md]
    )

    send_btn.click(
        fn=handle_chat,
        inputs=[user_query, chatbot, doc_filter, auto_route, top_k],
        outputs=[chatbot, user_query]
    )

    user_query.submit(
        fn=handle_chat,
        inputs=[user_query, chatbot, doc_filter, auto_route, top_k],
        outputs=[chatbot, user_query]
    )

    clear_btn.click(lambda: [], outputs=chatbot)

    download_dict_btn.click(
        fn=dict_out,
        inputs=None,
        outputs=download_file_output
    )

    download_chat_btn.click(
        fn=download_chat_history,
        inputs=chatbot,
        outputs=download_file_output
    )

demo.launch()

/Users/leo/Pfizer-RAG-AI-Pipeline/.venv/lib/python3.14/site-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
Loading weights: 100%|██████████████████████████████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 21103.02it/s]


Imports and configuration complete.
* Running on local URL:  http://127.0.0.1:7878
* To create a public link, set `share=True` in `launch()`.


Starting PDF extraction and analysis...
  Page 1: Cover Letter (page_in_doc=0)
  Page 2: Certificate Of Quality (page_in_doc=0)
  Page 3: Certificate Of Quality (page_in_doc=1)
  Page 4: Packaging Specification (page_in_doc=0)
  Page 5: Packaging Specification (page_in_doc=1)
  Page 6: Bse/Tse Declaration (page_in_doc=0)
  Page 7: Material Description (page_in_doc=0)
  Page 8: Supplier Qualification (page_in_doc=0)
  Page 9: Supplier Qualification (page_in_doc=1)
  Page 10: Chain Of Custody (page_in_doc=0)
Identified 7 logical documents:
   - Cover Letter: Pages 1-1
   - Certificate Of Quality: Pages 2-3
   - Packaging Specification: Pages 4-5
   - Bse/Tse Declaration: Pages 6-6
   - Material Description: Pages 7-7
   - Supplier Qualification: Pages 8-9
   - Chain Of Custody: Pages 10-10
Predicted document type: Packaging Specification
Predicted document type: Packaging Specification
